# Project 4: Music Popularity Prediction


This project will take data features collected for songs that have been on the Top 200 Weekly (Global) charts of Spotify in 2020 & 2021. The popularity of the song will be predicted using a tree-based regression model trained on these features.



The goals for the project are:

- Minimize the cross-validated ***root mean squared error ( RMSE )*** when predicting the popularity of a new song.

- Determine the importance of the features in driving the regression result.
The project will be done using tree-based regression techniques as covered in class. The hyperparameters of the trees should be carefully selected to avoid over-fitting.


There are three main challenges for this project:

1. Determining the outcome ( i.e. target ).  There is a "popularity" column.  But other columns may or may not be more appropriate indicators of popularity.

1. Choosing appropriate predictors ( i.e. features ). When building a machine learning model, we want to make sure that we consider how the model will be ultimately used. For this project, we are predicting the popularity of a new song. Therefore, we should only include the predictors we would have for a new song. It might help to imagine that the song will not be released for several weeks.

1. Data cleaning and feature engineering. Some creative cleaning and/or feature engineering may be needed to extract useful information for prediction.



Once again, be sure to go through the whole data science process and document as such in your Jupyter notebook.

The data is available on AWS S3 at https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv .



In [1]:
import numpy as np
import pandas as pd


In [2]:
# Set the URL
url = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv"
url


'https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv'

In [3]:
# Look at the headers
!curl -s -I {url}


HTTP/1.1 200 OK
x-amz-id-2: OfNVf/CGBnuKzr9cBFW26FdybnkyLy7k8MN2urTtehs3oPLwBELDVc/M53RpfHtzwBjT8mkSz1Jv/oxF0L50xH6Ppijw+eOB
x-amz-request-id: MCZVS2XJBV2D1HZ0
Date: Sat, 11 Jul 2026 18:20:18 GMT
Last-Modified: Wed, 04 Oct 2023 17:23:56 GMT
ETag: "65b9875b11e0d7ea03ee2af024f45e99"
x-amz-server-side-encryption: AES256
Accept-Ranges: bytes
Content-Type: text/csv
Content-Length: 738124
Server: AmazonS3



In [4]:
# Download the file
!curl -s -O {url}


In [5]:
# Verify
!ls -la


total 740
drwxr-xr-x 1 root root   4096 Jul 11 17:20 .
drwxr-xr-x 1 root root   4096 Jul 11 17:15 ..
drwxr-xr-x 4 root root   4096 Jun  4 13:32 .config
drwxr-xr-x 1 root root   4096 Jun  4 13:32 sample_data
-rw-r--r-- 1 root root 738124 Jul 11 18:20 Spotify.csv


In [6]:
# Look at the field names
!head -1 Spotify.csv | tr , '\n' | cat -n


     1	Index
     2	Highest Charting Position
     3	Number of Times Charted
     4	Week of Highest Charting
     5	Song Name
     6	Streams
     7	Artist
     8	Artist Followers
     9	Song ID
    10	Genre
    11	Release Date
    12	Weeks Charted
    13	Popularity
    14	Danceability
    15	Energy
    16	Loudness
    17	Speechiness
    18	Acousticness
    19	Liveness
    20	Tempo
    21	Duration (ms)
    22	Valence
    23	Chord


## MAKING SINCE OF  DATA

In [7]:
spotify = pd.read_csv( url, index_col = 0 )
spotify

,Highest Charting Position,Number of Times Charted,Week of Highest Charting,Song Name,Streams,Artist,Artist Followers,Song ID,Genre,Release Date,...,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
Index,,,,,,,,,,,,,,,,,,,,,
1,1,8,2021-07-23--2021-07-30,Beggin',"48,633,449",Måneskin,3377762,3Wrjm47oTz2sjIgck11l5e,"['indie rock italiano', 'italian pop']",2017-12-08,...,0.714,0.8,-4.808,0.0504,0.127,0.359,134.002,211560,0.589,B
2,2,3,2021-07-23--2021-07-30,STAY (with Justin Bieber),"47,248,719",The Kid LAROI,2230022,5HCyWlXZPP0y6Gqq8TgA20,['australian hip hop'],2021-07-09,...,0.591,0.764,-5.484,0.0483,0.0383,0.103,169.928,141806,0.478,C#/Db
3,1,11,2021-06-25--2021-07-02,good 4 u,"40,162,559",Olivia Rodrigo,6266514,4ZtFanR9U6ndgddUvNcjcG,['pop'],2021-05-21,...,0.563,0.664,-5.044,0.154,0.335,0.0849,166.928,178147,0.688,A
4,3,5,2021-07-02--2021-07-09,Bad Habits,"37,799,456",Ed Sheeran,83293380,6PQ88X9TkUIAUIZJHW2upE,"['pop', 'uk pop']",2021-06-25,...,0.808,0.897,-3.712,0.0348,0.0469,0.364,126.026,231041,0.591,B
5,5,1,2021-07-23--2021-07-30,INDUSTRY BABY (feat. Jack Harlow),"33,948,454",Lil Nas X,5473565,27NovPIUIRrOZoCHxABJwK,"['lgbtq+ hip hop', 'pop rap']",2021-07-23,...,0.736,0.704,-7.409,0.0615,0.0203,0.0501,149.995,212000,0.894,D#/Eb
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1552,195,1,2019-12-27--2020-01-03,New Rules,"4,630,675",Dua Lipa,27167675,2ekn2ttSfGqwhhate0LSR0,"['dance pop', 'pop', 'uk pop']",2017-06-02,...,0.762,0.7,-6.021,0.0694,0.00261,0.153,116.073,209320,0.608,A
1553,196,1,2019-12-27--2020-01-03,Cheirosa - Ao Vivo,"4,623,030",Jorge & Mateus,15019109,2PWjKmjyTZeDpmOUa3a5da,"['sertanejo', 'sertanejo universitario']",2019-10-11,...,0.528,0.87,-3.123,0.0851,0.24,0.333,152.37,181930,0.714,B
1554,197,1,2019-12-27--2020-01-03,Havana (feat. Young Thug),"4,620,876",Camila Cabello,22698747,1rfofaqEpACxVEHIZBJe6W,"['dance pop', 'electropop', 'pop', 'post-teen ...",2018-01-12,...,0.765,0.523,-4.333,0.03,0.184,0.132,104.988,217307,0.394,D


In [8]:
spotify.transpose()

Index,1,2,3,4,5,6,7,8,9,10,...,1547,1548,1549,1550,1551,1552,1553,1554,1555,1556
Highest Charting Position,1,2,1,3,5,1,3,2,3,8,...,143,156,178,187,190,195,196,197,198,199
Number of Times Charted,8,3,11,5,1,18,16,10,8,10,...,1,1,1,1,1,1,1,1,1,1
Week of Highest Charting,2021-07-23--2021-07-30,2021-07-23--2021-07-30,2021-06-25--2021-07-02,2021-07-02--2021-07-09,2021-07-23--2021-07-30,2021-05-07--2021-05-14,2021-05-14--2021-05-21,2021-06-18--2021-06-25,2021-06-18--2021-06-25,2021-07-02--2021-07-09,...,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03
Song Name,Beggin',STAY (with Justin Bieber),good 4 u,Bad Habits,INDUSTRY BABY (feat. Jack Harlow),MONTERO (Call Me By Your Name),Kiss Me More (feat. SZA),Todo De Ti,Yonaguni,I WANNA BE YOUR SLAVE,...,JACKBOYS,Combatchy (feat. MC Rebecca),Old Town Road,Let Me Know (I Wonder Why Freestyle),Ne reviens pas,New Rules,Cheirosa - Ao Vivo,Havana (feat. Young Thug),Surtada - Remix Brega Funk,Lover (Remix) [feat. Shawn Mendes]
Streams,"48,633,449","47,248,719","40,162,559","37,799,456","33,948,454","30,071,134","29,356,736","26,951,613","25,030,128","24,551,591",...,"5,363,493","5,149,797","4,852,004","4,701,532","4,676,857","4,630,675","4,623,030","4,620,876","4,607,385","4,595,450"
Artist,Måneskin,The Kid LAROI,Olivia Rodrigo,Ed Sheeran,Lil Nas X,Lil Nas X,Doja Cat,Rauw Alejandro,Bad Bunny,Måneskin,...,JACKBOYS,"Anitta, Lexa, Luísa Sonza",Lil Nas X,Juice WRLD,"Gradur, Heuss L'enfoiré",Dua Lipa,Jorge & Mateus,Camila Cabello,"Dadá Boladão, Tati Zaqui, OIK",Taylor Swift
Artist Followers,3377762,2230022,6266514,83293380,5473565,5473565,8640063,6080597,36142273,3377762,...,437907,10741972,5488666,19102888,1390813,27167675,15019109,22698747,208630,42227614
Song ID,3Wrjm47oTz2sjIgck11l5e,5HCyWlXZPP0y6Gqq8TgA20,4ZtFanR9U6ndgddUvNcjcG,6PQ88X9TkUIAUIZJHW2upE,27NovPIUIRrOZoCHxABJwK,67BtfxlNbhBmCDR2L2l8qd,748mdHapucXQri7IAO8yFK,4fSIb4hdOQ151TILNsSEaF,2JPLbjOn0wPCngEot2STUS,4pt5fDVTg5GhEvEtlz9dKk,...,62zKJrpbLxz6InR3tGyr7o,2bPtwnrpFNEe8N7Q85kLHw,2YpeDb67231RjR0MgVLzsG,3wwo0bJvDSorOpNfzEkfXx,4TnFANpjVwVKWzkxNzIyFH,2ekn2ttSfGqwhhate0LSR0,2PWjKmjyTZeDpmOUa3a5da,1rfofaqEpACxVEHIZBJe6W,5F8ffc8KWKNawllr5WsW0r,3i9UVldZOE0aD0JnyfAZZ0
Genre,"['indie rock italiano', 'italian pop']",['australian hip hop'],['pop'],"['pop', 'uk pop']","['lgbtq+ hip hop', 'pop rap']","['lgbtq+ hip hop', 'pop rap']","['dance pop', 'pop']","['puerto rican pop', 'trap latino']","['latin', 'reggaeton', 'trap latino']","['indie rock italiano', 'italian pop']",...,"['rap', 'trap']","['funk carioca', 'funk pop', 'pagode baiano', ...","['lgbtq+ hip hop', 'pop rap']","['chicago rap', 'melodic rap']","['francoton', 'french hip hop', 'pop urbaine',...","['dance pop', 'pop', 'uk pop']","['sertanejo', 'sertanejo universitario']","['dance pop', 'electropop', 'pop', 'post-teen ...","['brega funk', 'funk carioca']","['pop', 'post-teen pop']"
Release Date,2017-12-08,2021-07-09,2021-05-21,2021-06-25,2021-07-23,2021-03-31,2021-04-09,2021-05-20,2021-06-04,2021-03-19,...,2019-12-27,2019-11-20,2019-06-21,2019-12-07,2019-11-29,2017-06-02,2019-10-11,2018-01-12,2019-09-25,2019-11-13


In [9]:
spotify.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1556 entries, 1 to 1556
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Highest Charting Position  1556 non-null   int64 
 1   Number of Times Charted    1556 non-null   int64 
 2   Week of Highest Charting   1556 non-null   object
 3   Song Name                  1556 non-null   object
 4   Streams                    1556 non-null   object
 5   Artist                     1556 non-null   object
 6   Artist Followers           1556 non-null   object
 7   Song ID                    1556 non-null   object
 8   Genre                      1556 non-null   object
 9   Release Date               1556 non-null   object
 10  Weeks Charted              1556 non-null   object
 11  Popularity                 1556 non-null   object
 12  Danceability               1556 non-null   object
 13  Energy                     1556 non-null   object
 14  Loudness     

In [10]:
# there are no nulls
spotify.isna().sum().sort_values(ascending=False).mul(1)
# the reason for this is because they are all objects even if it has a space it claims it to be information

,0
Highest Charting Position,0
Number of Times Charted,0
Week of Highest Charting,0
Song Name,0
Streams,0
Artist,0
Artist Followers,0
Song ID,0
Genre,0
Release Date,0


In [11]:
nunique = spotify.nunique().sort_values( ascending = False )
nunique
# seems like columns 'song name', ' streams ', song ID '. are very high in non uniqueness meaning that they are identifiers.
# for this project since we are trying to predict what makes a popular song those columns will not help with testing and running a prediction model

,0
Streams,1556
Song Name,1556
Song ID,1517
Duration (ms),1486
Tempo,1461
Loudness,1394
Acousticness,965
Weeks Charted,775
Speechiness,772
Valence,732


In [12]:
# popularity id full of whole numbers yet it is an object. it is out target. lets convert to int.
#spotify['Popularity'] = pd.to_numeric(spotify['Popularity'], errors='coerce').astype(int)

# Verify the fix
#spotify['Popularity'].dtype

In [13]:
# was not able to be done yet lets convert all NAN and blank sopts into a 0

In [14]:
# Convert to numeric
spotify['Popularity'] = pd.to_numeric(spotify['Popularity'], errors='coerce')

# Fill any NaN values with 0, then cast to regular int
spotify['Popularity'] = spotify['Popularity'].fillna(0).astype(int)

# Verify the fix
print(spotify['Popularity'].dtype)

int64


In [15]:
print(spotify['Popularity'])

Index
1       100
2        99
3        99
4        98
5        96
       ... 
1552     79
1553     66
1554     81
1555     60
1556     70
Name: Popularity, Length: 1556, dtype: int64


In [16]:
spotify['Popularity'].isna()
# of course non of them are gonna be NAN/nulls thats because I literally changed the value to 0 for those spots

,Popularity
Index,
1,False
2,False
3,False
4,False
5,False
...,...
1552,False
1553,False
1554,False


In [17]:
# im trying to just choose what makes it a 100-80 range of popularity. not trying to figure out what makes a song not popular
# just want to predict if it good. whats bad whould be the opposite of my findings

In [18]:
(spotify['Popularity'] < 80).sum()

np.int64(1164)

## Data Cleaning

In [21]:
spotify_c = spotify.copy()

In [22]:
# dropping the following I determined them to be non important to predicticting a non-released song as a future hit song
# 'Artist' information is bias aswell as 'Artist Followers'
# Song Name, Song ID, Streams are identifiers
# Danceabillity is something that is recoreded after the fact of release not before so thats dropped. thats something that we can also try predicting not something thatll help us predict.

spotify_c = spotify_c.drop( columns = ['Song Name','Streams', 'Song ID', 'Danceability', 'Artist Followers', 'Artist', 'Release Date',])



In [23]:
spotify_c.shape

(1556, 15)

In [24]:
# I have a lot of Objects that I must change into floats. floats because they are nubers with decimals.
# some of those fields have information. those blank spots must be filled. and using gemini I found that I can use 'coerce'
spotify_c.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1556 entries, 1 to 1556
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Highest Charting Position  1556 non-null   int64 
 1   Number of Times Charted    1556 non-null   int64 
 2   Week of Highest Charting   1556 non-null   object
 3   Genre                      1556 non-null   object
 4   Weeks Charted              1556 non-null   object
 5   Popularity                 1556 non-null   int64 
 6   Energy                     1556 non-null   object
 7   Loudness                   1556 non-null   object
 8   Speechiness                1556 non-null   object
 9   Acousticness               1556 non-null   object
 10  Liveness                   1556 non-null   object
 11  Tempo                      1556 non-null   object
 12  Duration (ms)              1556 non-null   object
 13  Valence                    1556 non-null   object
 14  Chord        

In [27]:
# Define the list of columns you want to change to float. chose those because they are decimal values
float_cols = [
    'Energy', 'Loudness', 'Speechiness', 'Acousticness',
    'Liveness', 'Tempo', 'Duration (ms)', 'Valence'
]

# 2. Convert them all to numeric floats
spotify_c[float_cols] = spotify_c[float_cols].apply(pd.to_numeric, errors='coerce')

# 3. Verify the changes
spotify_c.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1556 entries, 1 to 1556
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Highest Charting Position  1556 non-null   int64  
 1   Number of Times Charted    1556 non-null   int64  
 2   Week of Highest Charting   1556 non-null   object 
 3   Genre                      1556 non-null   object 
 4   Weeks Charted              1556 non-null   object 
 5   Popularity                 1556 non-null   int64  
 6   Energy                     1545 non-null   float64
 7   Loudness                   1545 non-null   float64
 8   Speechiness                1545 non-null   float64
 9   Acousticness               1545 non-null   float64
 10  Liveness                   1545 non-null   float64
 11  Tempo                      1545 non-null   float64
 12  Duration (ms)              1545 non-null   float64
 13  Valence                    1545 non-null   float64
 1

In [26]:
spotify_c.transpose()

Index,1,2,3,4,5,6,7,8,9,10,...,1547,1548,1549,1550,1551,1552,1553,1554,1555,1556
Highest Charting Position,1,2,1,3,5,1,3,2,3,8,...,143,156,178,187,190,195,196,197,198,199
Number of Times Charted,8,3,11,5,1,18,16,10,8,10,...,1,1,1,1,1,1,1,1,1,1
Week of Highest Charting,2021-07-23--2021-07-30,2021-07-23--2021-07-30,2021-06-25--2021-07-02,2021-07-02--2021-07-09,2021-07-23--2021-07-30,2021-05-07--2021-05-14,2021-05-14--2021-05-21,2021-06-18--2021-06-25,2021-06-18--2021-06-25,2021-07-02--2021-07-09,...,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03
Genre,"['indie rock italiano', 'italian pop']",['australian hip hop'],['pop'],"['pop', 'uk pop']","['lgbtq+ hip hop', 'pop rap']","['lgbtq+ hip hop', 'pop rap']","['dance pop', 'pop']","['puerto rican pop', 'trap latino']","['latin', 'reggaeton', 'trap latino']","['indie rock italiano', 'italian pop']",...,"['rap', 'trap']","['funk carioca', 'funk pop', 'pagode baiano', ...","['lgbtq+ hip hop', 'pop rap']","['chicago rap', 'melodic rap']","['francoton', 'french hip hop', 'pop urbaine',...","['dance pop', 'pop', 'uk pop']","['sertanejo', 'sertanejo universitario']","['dance pop', 'electropop', 'pop', 'post-teen ...","['brega funk', 'funk carioca']","['pop', 'post-teen pop']"
Weeks Charted,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,2021-07-23--2021-07-30,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,...,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03,2019-12-27--2020-01-03
Popularity,100,99,99,98,96,97,94,95,96,95,...,56,64,81,76,62,79,66,81,60,70
Energy,0.8,0.764,0.664,0.897,0.704,0.508,0.701,0.718,0.648,0.608,...,0.13,0.73,0.619,0.537,0.778,0.7,0.87,0.523,0.55,0.603
Loudness,-4.808,-5.484,-5.044,-3.712,-7.409,-6.682,-3.541,-3.605,-4.601,-4.008,...,-25.166,-3.032,-5.56,-7.895,-3.384,-6.021,-3.123,-4.333,-7.026,-7.176
Speechiness,0.0504,0.0483,0.154,0.0348,0.0615,0.152,0.0286,0.0506,0.118,0.0387,...,0.0336,0.0809,0.102,0.0832,0.0638,0.0694,0.0851,0.03,0.0587,0.064
Acousticness,0.127,0.0383,0.335,0.0469,0.0203,0.297,0.235,0.31,0.276,0.00165,...,0.9,0.383,0.0533,0.172,0.212,0.00261,0.24,0.184,0.249,0.433
